## Phase 1: Environment Setup & GPU Verification

In [1]:
# 1. Fix PyG dependencies to match the HPC's Torch version
import torch
v = torch.__version__.split('+')[0]
cu = "cu" + torch.version.cuda.replace('.', '')
print(f"Patching PyG for Torch {v} on {cu}...")
!pip install torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-{v}+{cu}.html --quiet

import os
import sys
import pandas as pd
import numpy as np

# 2. Bridge to the root directory
sys.path.append(os.path.abspath('..'))

# 3. Corrected Imports from physics_config
# Note: 'RespiratoryCostant' has the typo from your source file
from gnn.model import STPIGNN
from shared.physics_config import PHYSICS_LOSS_LAMBDA, RespiratoryCostant, get_respiratory_minute_volume

# 4. Verify V100 Hardware & VRAM
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"Connected to: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {props.total_memory / 1024**3:.2f} GB")
else:
    print("CUDA NOT FOUND. Check kernel.")

print(f"Physics Lambda: {PHYSICS_LOSS_LAMBDA}")
print(f"Cycling Inhalation Rate: {RespiratoryCostant.CYCLING} m^3/hr")

Patching PyG for Torch 2.8.0 on cu128...


/home/jupyter-1nt23cb058/.local/lib/python3.9/site-packages/torch_geometric/typing.py:86: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /lib/x86_64-linux-gnu/libc.so.6: version `GLIBC_2.32' not found (required by /home/jupyter-1nt23cb058/.local/lib/python3.9/site-packages/torch_scatter/_version_cuda.so)
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
/home/jupyter-1nt23cb058/.local/lib/python3.9/site-packages/torch_geometric/typing.py:97: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: /lib/x86_64-linux-gnu/libc.so.6: version `GLIBC_2.32' not found (required by /home/jupyter-1nt23cb058/.local/lib/python3.9/site-packages/torch_cluster/_version_cuda.so)
  warnings.warn(f"An issue occurred while importing 'torch-cluster'. "
/home/jupyter-1nt23cb058/.local/lib/python3.9/site-packages/torch_geometric/typing.py:113: UserWarning: An issue occurred while importing 'torch-s

Connected to: Tesla V100-PCIE-32GB
Total VRAM: 31.74 GB
Physics Lambda: 0.1
Cycling Inhalation Rate: 3.5 m^3/hr


## Phase 2: Data Ingestion (155k-Node Graph & Features)

In [2]:
import torch
import pandas as pd
import os

# 1. Device and Path Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GRAPH_PATH = '../data/processed/static_graph_pyg.pt'
FEATURE_PATH = '../data/processed/gnn_training_tensor_final.parquet'

# 2. Load the 155k-Node Static Graph
print("Loading static graph to V100...")
graph = torch.load(GRAPH_PATH, map_location=device, weights_only=False)

# 3. Load Multi-Mode Feature Parquet
print("Loading multi-mode features...")
df_features = None

try:
    # Attempt loading with fastparquet to bypass PyArrow extension issues
    df_features = pd.read_parquet(FEATURE_PATH, engine='fastparquet')
except Exception as e:
    print(f"Fastparquet/Standard read failed, trying pyarrow table: {e}")
    try:
        import pyarrow.parquet as pq
        df_features = pq.read_table(FEATURE_PATH).to_pandas()
    except Exception as e2:
        print(f"Critical Error: PyArrow also failed: {e2}")

# 4. The Big Merge: Aligning 1.9M Nodes to the GNN
if df_features is not None:
    print("df_features successfully loaded.")
    
    # Identify and drop non-numeric metadata
    cols_to_drop = ['node_id', 'timestamp', 'station_id', 'station_name']
    feature_matrix = df_features.drop(columns=cols_to_drop, errors='ignore').values

    # Push Node Features to V100 GPU
    graph.x = torch.tensor(feature_matrix, dtype=torch.float).to(device)

    # Map Ground Truth (PM2.5) to graph.y
    if 'station_pm25' in df_features.columns:
        graph.y = torch.tensor(df_features['station_pm25'].values, dtype=torch.float).to(device)
    
    print("-" * 30)
    print(f"Node Features (graph.x) Shape: {graph.x.shape}")
    print(f"Edge Attributes (Road Types) Shape: {graph.edge_attr.shape}")
    print(f"VRAM Occupied: {torch.cuda.memory_allocated(device) / 1024**2:.2f} MB")
else:
    print("Error: df_features was not defined. The training matrix cannot be built.")

Loading static graph to V100...
Loading multi-mode features...
df_features successfully loaded.
------------------------------
Node Features (graph.x) Shape: torch.Size([1932700, 17])
Edge Attributes (Road Types) Shape: torch.Size([424550, 29])
VRAM Occupied: 186.97 MB


## Phase 3: ST-PIGNN Model Initialization (V100 Optimized)

In [3]:
## Phase 3: ST-PIGNN Model Initialization

import torch
from gnn.model import STPIGNN

# 1. Configuration based on your Model Signature
# node_in_dim: 17 (Environmental/CAMS data)
# edge_dim: 29 (Road-type infrastructure for the 4 modes)
NODE_IN = graph.x.shape[1]
EDGE_IN = graph.edge_attr.shape[1]

# We can now independently tune Spatial vs Temporal hidden dims
# Using 256 for both to utilize the V100's capacity
SPATIAL_HIDDEN = 256
TEMPORAL_HIDDEN = 256
GNN_LAYERS = 3       # Deeper spatial reasoning for complex Bangalore roads
GRU_LAYERS = 2       # Better temporal persistence for the 1.9M nodes
DROPOUT = 0.1

# 2. Initialize Model with exact keyword arguments
model = STPIGNN(
    node_in_dim=NODE_IN,
    edge_dim=EDGE_IN,
    spatial_hidden_dim=SPATIAL_HIDDEN,
    temporal_hidden_dim=TEMPORAL_HIDDEN,
    gnn_layers=GNN_LAYERS,
    gru_layers=GRU_LAYERS,
    dropout=DROPOUT
).to(device)

# 3. Optimization Suite
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scaler = torch.cuda.amp.GradScaler() 

print("-" * 30)
print(f"ST-PIGNN Initialized on {device}")
print(f"Architecture: {GNN_LAYERS} GNN Layers + {GRU_LAYERS} GRU Layers")
print(f"Total Trainable Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"VRAM Occupied: {torch.cuda.memory_allocated(device) / 1024**2:.2f} MB")

------------------------------
ST-PIGNN Initialized on cuda
Architecture: 3 GNN Layers + 2 GRU Layers
Total Trainable Parameters: 1,213,697
VRAM Occupied: 191.60 MB


/tmp/ipykernel_9576/837805733.py:33: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


## Phase 4: Exposure-Aware Loss Function (Active Commuter Logic)

In [4]:
## Phase 4: Exposure-Aware Loss Function

import torch.nn.functional as F

def compute_hybrid_loss(out, target, mask):
    """
    Combines Data-Driven Accuracy with Physical Constraints.
    
    Args:
        out: The GNN's PM2.5 prediction for all 1.93M nodes.
        target: The ground truth PM2.5 from the 23 sensors (graph.y).
        mask: 'graph.train_mask' identifying which nodes are actual sensors.
    """
    
    # 1. Supervised Data Loss (MSE)
    # This forces the GNN to match the actual sensor readings in Bangalore.
    if mask is not None and mask.sum() > 0:
        # out[mask] extracts only the nodes that are physical sensors
        data_loss = F.mse_loss(out[mask], target[mask])
    else:
        # Fallback if the mask hasn't been initialized
        data_loss = torch.tensor(0.0).to(device)
    
    # 2. Physics-Informed Regularization (Module 2 Hook)
    # In Phase 5, we can integrate the physics_loss here.
    # This ensures that even if a Jogging path has no sensor, the 
    # model follows dispersion laws based on the 29 edge attributes.
    
    # physics_loss = model.compute_physics_loss(graph, out)
    # return data_loss + (PHYSICS_LOSS_LAMBDA * physics_loss)
    
    return data_loss

print("Hybrid Loss Function defined and anchored to 23 sensors.")

Hybrid Loss Function defined and anchored to 23 sensors.


## Phase 5: VRAM Cleanup and  Data Streaming Setup

In [5]:
import torch
import gc
import os

# 1. Safety Loader: Re-load graph if the kernel was restarted
GRAPH_PATH = '../data/processed/static_graph_pyg.pt'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if 'graph' not in locals():
    print("🔄 Graph not found in memory. Re-loading from disk...")
    graph = torch.load(GRAPH_PATH, map_location='cpu', weights_only=False)
else:
    # If it exists, push features to CPU to save GPU VRAM
    graph.x = graph.x.cpu()
    graph.y = graph.y.cpu()

# 2. Force clear VRAM from any failed runs
if 'model' in locals(): del model
if 'optimizer' in locals(): del optimizer
if 'pred' in locals(): del pred

gc.collect()
torch.cuda.empty_cache()

# 3. Setup CPU Streaming tensors
# We keep 1.9M rows on CPU and only "stream" 1 hour to the GPU at a time
N_NODES = 154902 
T_STEPS = graph.x.shape[0] // N_NODES

x_seq_cpu = graph.x[:T_STEPS * N_NODES].view(T_STEPS, N_NODES, -1)
y_seq_cpu = graph.y[:T_STEPS * N_NODES].view(T_STEPS, N_NODES)
train_mask_gpu = graph.train_mask.to(device)

print("-" * 30)
print(f"VRAM Cleared. GPU 0 memory reset.")
print(f"Ready to stream {T_STEPS} timesteps from CPU to {device}")

------------------------------
VRAM Cleared. GPU 0 memory reset.
Ready to stream 12 timesteps from CPU to cuda


## Phase 6: Model & Optimizer Re-Initialization

In [6]:
import types
import torch.nn.functional as F
from gnn.model import STPIGNN

# 1. Initialize Model & Projection
input_projection = torch.nn.Linear(17, 256).to(device)
model = STPIGNN(
    node_in_dim=17, 
    edge_dim=29,    
    spatial_hidden_dim=256,
    temporal_hidden_dim=256,
    gnn_layers=3,
    gru_layers=2
).to(device)

# 2. Apply Python 3.9 Compatibility Patch
def patched_spatial_encode(self, x_t, edge_index, edge_attr):
    x = x_t.view(-1, x_t.size(-1))
    x = input_projection(x)
    for conv, norm in zip(self.gnn_layers, self.norms):
        residual = x
        x = conv(x, edge_index, edge_attr)
        x = norm(x)
        x = F.relu(x)
        p_val = self.dropout.p if isinstance(self.dropout, torch.nn.Dropout) else self.dropout
        x = F.dropout(x, p=p_val, training=self.training)
        if x.shape == residual.shape: x = x + residual
    return x

model._spatial_encode = types.MethodType(patched_spatial_encode, model)
optimizer = torch.optim.Adam(list(model.parameters()) + list(input_projection.parameters()), lr=0.001)
scaler = torch.amp.GradScaler('cuda')

print("Model patched and ready for V100 optimized execution.")

Model patched and ready for V100 optimized execution.


## Phase 7: Spatio-Temporal Stochastic Optimization & State Preservation

In [ ]:
from tqdm.auto import tqdm
import os
import sys

# 1. Persistence Configuration
CHECKPOINT_PATH = '../cache/stpignn_checkpoint.tar'
TEMP_CHECKPOINT = '../cache/stpignn_checkpoint_tmp.tar'
TOTAL_EPOCHS = 100
start_epoch = 1

# 2. Interactive State Recovery & Pre-Flight Check
print("🔍 --- Pre-Flight System Check ---")
if os.path.exists(CHECKPOINT_PATH):
    # Map to CPU first to avoid fragmentation during loading
    ckpt = torch.load(CHECKPOINT_PATH, map_location='cpu')
    last_epoch = ckpt['epoch']
    last_loss = ckpt.get('loss', 0.0)
    print(f"📂 Found existing state: Epoch {last_epoch} | MSE: {last_loss:.6f}")
    
    user_input = input(f"❓ Resume from Epoch {last_epoch + 1}? (yes/no): ").lower().strip()
    if user_input == 'yes':
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        input_projection.load_state_dict(ckpt['proj_state_dict'])
        scaler.load_state_dict(ckpt['scaler_state_dict'])
        start_epoch = last_epoch + 1
        print(f"🚀 State recovered. Resuming at Epoch {start_epoch}...")
    else:
        print("🆕 Starting fresh sequence.")
else:
    print("🆕 No existing state found.")
    user_input = input("❓ Start fresh training? (yes/no): ").lower().strip()
    if user_input != 'yes':
        print("🛑 Execution aborted by user.")
        # Using a soft return instead of sys.exit to keep the kernel alive
        user_input = 'abort'

# 3. Final Hardware Verification
if user_input == 'yes':
    vram_free = torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()
    print(f"✅ VRAM Available: {vram_free / 1024**3:.2f} GB")
    print(f"✅ Training Target: {TOTAL_EPOCHS} Epochs")
    print("-" * 30)

    # 4. Optimization Loop with Dual Progress Monitoring
    try:
        # Global Progress Bar (The 100 Epochs)
        global_pbar = tqdm(range(start_epoch, TOTAL_EPOCHS + 1), desc="Overall Training", position=0)
        
        for epoch in global_pbar:
            model.train()
            input_projection.train()
            epoch_loss = 0
            
            # Inner Progress Bar (The 12-hour sequence)
            # unit="hr" helps track Bangalore's temporal data chunks
            inner_pbar = tqdm(range(T_STEPS), desc=f"Epoch {epoch}", unit="hr", position=1, leave=False)
            
            for t in inner_pbar:
                optimizer.zero_grad(set_to_none=True) 
                
                # Window Streaming (T=1)
                x_win = x_seq_cpu[t:t+1].unsqueeze(0).to(device)
                y_win = y_seq_cpu[t:t+1].to(device)
                
                with torch.amp.autocast('cuda'):
                    # Forward Pass: GINEConv Spatial + GRU Temporal
                    pred = model(x_win, graph.edge_index.to(device), graph.edge_attr.to(device))
                    pred_flat = pred.squeeze()
                    
                    # Loss evaluated only at the 23 sensor coordinates
                    loss = F.mse_loss(pred_flat[train_mask_gpu], y_win.squeeze()[train_mask_gpu])
                
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                
                epoch_loss += loss.item()
                inner_pbar.set_postfix({"MSE": f"{loss.item():.4f}"})
                
                del x_win, y_win, pred
                torch.cuda.empty_cache()

            # Atomic State Preservation
            avg_loss = epoch_loss / T_STEPS
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'proj_state_dict': input_projection.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scaler_state_dict': scaler.state_dict(),
                'loss': avg_loss
            }, TEMP_CHECKPOINT)
            os.replace(TEMP_CHECKPOINT, CHECKPOINT_PATH)
            
            # Update Global Bar with Convergence Stats
            global_pbar.set_postfix({"Last MSE": f"{avg_loss:.6f}"})

        print(f"\n✅ Training Complete. Model state secured in {CHECKPOINT_PATH}")

    except KeyboardInterrupt:
        print("\n🛑 Execution suspended. Your progress is safe. Catch your ride!")
else:
    if user_input != 'abort':
        print("Session inactive. Re-run cell to engage training sequence.")